In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import datasets, layers, models
from tensorflow.keras.utils import to_categorical

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)


In [ ]:
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

print("Train images:", train_images.shape, "| Train labels:", train_labels.shape)
print("Test images :", test_images.shape, "| Test labels :", test_labels.shape)


In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5,5,i+1)
    plt.xticks([]); plt.yticks([]); plt.grid(False)
    plt.imshow(train_images[i])
    plt.xlabel(class_names[train_labels[i][0]])
plt.suptitle("Sample Images from CIFAR-10")
plt.show()


In [ ]:
import collections
counts = collections.Counter(train_labels.flatten())
plt.figure(figsize=(8,4))
plt.bar([class_names[i] for i in counts.keys()], counts.values())
plt.title('Class Distribution (Training Set)')
plt.xticks(rotation=45)
plt.ylabel('Count')
plt.show()


In [ ]:
# Normalize pixel values from [0, 255] to [0, 1]
train_images_norm = train_images / 255.0
test_images_norm = test_images / 255.0

# One-hot encode labels (needed for categorical_crossentropy)
train_labels_cat = to_categorical(train_labels, num_classes=10)
test_labels_cat = to_categorical(test_labels, num_classes=10)

print("Labels example before encoding:", train_labels[0])
print("Labels example after encoding :", train_labels_cat[0])


In [ ]:
def build_model_1():
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model_1 = build_model_1()
model_1.summary()


In [ ]:
history_1 = model_1.fit(
    train_images_norm, train_labels_cat,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    verbose=1
)


In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12,4))
    axes[0].plot(history.history['loss'], label='train')
    axes[0].plot(history.history['val_loss'], label='val')
    axes[0].set_title(f'{title} - Loss')
    axes[0].set_xlabel('Epoch'); axes[0].legend()

    axes[1].plot(history.history['accuracy'], label='train')
    axes[1].plot(history.history['val_accuracy'], label='val')
    axes[1].set_title(f'{title} - Accuracy')
    axes[1].set_ylim([0, 1])
    axes[1].set_xlabel('Epoch'); axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(history_1, 'Model 1 (Baseline)')


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

def evaluate_model(model, X_test, y_test_cat, y_test_raw, name='Model'):
    test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
    print(f"--- {name} ---")
    print(f"Test Loss    : {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")

    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    y_true = y_test_raw.flatten()

    print()
    print(classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(9,9))
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
    plt.title(f'{name} - Confusion Matrix')
    plt.show()

    return {'model': name, 'test_loss': test_loss, 'test_accuracy': test_acc}

results = []
results.append(evaluate_model(model_1, test_images_norm, test_labels_cat, test_labels, 'Model 1 (Baseline)'))


In [ ]:
def build_model_2():
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),

        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model_2 = build_model_2()
model_2.summary()


In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

history_2 = model_2.fit(
    train_images_norm, train_labels_cat,
    validation_split=0.1,
    epochs=25,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
plot_history(history_2, 'Model 2 (Deeper+BatchNorm+Dropout)')


In [ ]:
results.append(evaluate_model(model_2, test_images_norm, test_labels_cat, test_labels, 'Model 2 (Deeper+BatchNorm+Dropout)'))


In [ ]:
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)
datagen.fit(train_images_norm)

model_3 = build_model_2() 

train_gen = datagen.flow(train_images_norm, train_labels_cat, batch_size=128)

history_3 = model_3.fit(
    train_gen,
    validation_data=(test_images_norm, test_labels_cat),
    epochs=25,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
plot_history(history_3, 'Model 3 (Data Augmentation)')


In [ ]:
results.append(evaluate_model(model_3, test_images_norm, test_labels_cat, test_labels, 'Model 3 (DataAugmentation)'))


In [ ]:
import pandas as pd
results_df = pd.DataFrame(results).set_index('model')
results_df.round(4)


In [ ]:
results_df['test_accuracy'].plot(kind='bar', figsize=(8,5), color='steelblue', legend=False)
plt.title('Test Accuracy Comparison Across Models')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
